::: {.callout-important}
## Main idea
Differentiating a model means chaining the derivatives of its parts. Reverse-mode automatic differentiation (backpropagation) does exactly that, automatically. The adjoint method of PDE-constrained optimization is the same idea applied to one special part: a state defined by *solving* an equation, whose derivative rule turns out to be a linear solve.
:::

We build up in three steps:

- A model is a composition of functions, so its gradient is a chain of local derivatives (plain autodiff).
- A solver that marches in time is such a composition, so backprop through it is the adjoint recurrence.
- A solver that reaches a steady state defines its state *implicitly*; that link's derivative rule is a linear solve, and that solve is the classical adjoint method (MacAyeal 1993).

(This is a composition-first retelling of the companion lecture. The destination, "backprop is the adjoint," is the same.)

In [1]:
import jax, jax.numpy as jnp, numpy as np, diffrax as dfx
jax.config.update("jax_enable_x64", True)
print("jax", jax.__version__, "| diffrax", dfx.__version__)

jax 0.10.2 | diffrax 0.7.2


## 1 · A model is a composition of functions

Write the model as a chain of operations. The **states** $x_k$ are what the model computes (activations in a network, or a physical field such as velocity or thickness in a solver). The **parameters** $\theta$ are what we differentiate with respect to (network weights, or a physical parameter such as a sliding coefficient $\beta$):

$$x_0 = \text{input},\qquad x_{k+1} = f_k(x_k,\theta)\ \ (k=0,\dots,N-1),\qquad L = \ell(x_0,x_1,\dots,x_N)\in\mathbb{R}.$$

Every state, and therefore the loss, is a function of $\theta$ through the chain. We let the loss depend on the *whole trajectory* (a data misfit summed over time, say); the familiar case $L=\ell(x_N)$ is just the version where only the last term is present.

::: {.callout-note}
## What $x$ and $\theta$ are
- *Neural network*: $x_k$ are the layer activations, $\theta$ the weights and biases, $L$ the training loss.
- *Glacier inversion*: $x_k$ is the solver's state (velocity, thickness), $\theta$ a parameter such as the sliding coefficient $\beta$, $L$ the misfit to observations.
:::

**The chain rule.** Each step feeds $\theta$ in two ways: directly through its $\theta$ argument, and indirectly through $x_k$, which already depends on $\theta$. Differentiating the whole model therefore just chains the local derivatives of its parts. Two ingredients are enough:

- the *local* derivatives of each operation: its Jacobian $J_k := \partial f_k/\partial x_k$, and its parameter-derivative $\partial f_k/\partial\theta$;
- a *rule* to compose them: the chain rule, which strings the $J_k$ into a product (the sensitivity of $x_N$ to $x_0$ is $J_{N-1}\cdots J_0$) and adds each step's $\theta$-contribution.

Automatic differentiation supplies both. The next section makes it concrete: the chain can be swept in two directions, and that choice is the whole story of efficiency.

::: {.callout-note}
## Why not finite differences or symbolic?
- *Finite differences* cost one extra evaluation per parameter, and force a step-size tradeoff (truncation vs. round-off).
- *Symbolic* differentiation of a long program suffers expression swell.
- *Automatic differentiation* sidesteps both, and this notebook is what it computes under the hood.
:::

## 2 · Forward and reverse mode

The chain from §1 can be swept in either direction. Both build the same gradient; they differ in what they propagate and which way.

### Forward mode: push sensitivities forward

Carry the *sensitivity* $t_k := dx_k/d\theta$ from the input toward the output. Each step passes the running sensitivity through its Jacobian and adds its own direct $\theta$-dependence:

$$t_0 = 0,\qquad t_{k+1} = J_k\,t_k + \frac{\partial f_k}{\partial\theta},\qquad \frac{dL}{d\theta} = \sum_{k=0}^{N}\frac{\partial L}{\partial x_k}\,t_k.$$

(This is just the derivative of one step $x_{k+1}=f_k(x_k,\theta)$: the chain rule through $x_k$, plus the direct $\theta$ term.)

- Runs front to back, alongside the values, so no history is stored ($O(1)$ extra memory).
- $t_k$ carries one column per parameter, so the cost scales with the number of parameters.
- One operation at a time this is a *Jacobian-vector product* (JVP, `jax.jvp`): push a *tangent* through.

### Reverse mode: pull sensitivities back

Carry the *adjoint* $\lambda_k$ (the loss's sensitivity to $x_k$) from the output back toward the input:

$$\lambda_N = \Big(\frac{\partial L}{\partial x_N}\Big)^{\!\top},\qquad \lambda_k = \Big(\frac{\partial L}{\partial x_k}\Big)^{\!\top} + J_k^{\top}\lambda_{k+1},\qquad \frac{dL}{d\theta} = \sum_{k=0}^{N-1}\lambda_{k+1}^{\top}\,\frac{\partial f_k}{\partial\theta}.$$

- Runs back to front, after a forward pass whose states must be kept (hence checkpointing, §5).
- For a scalar loss, one backward pass gives the whole gradient, regardless of the number of parameters.
- One operation at a time this is a *vector-Jacobian product* (VJP, `jax.vjp`): pull a *cotangent* back.

::: {.callout-note}
## Where the recurrences come from
Both fall out of one object. Stack the states $U=(x_0,\dots,x_N)$ and write each step as a constraint $g_{k+1}=x_{k+1}-f_k(x_k,\theta)=0$. Each step touches only the previous one, so $\partial g/\partial U$ is block lower-triangular ($I$ on the diagonal, $-J_k$ just below):

$$\frac{\partial g}{\partial U}=
\begin{pmatrix}
I & & & \\
-J_0 & I & & \\
& \ddots & \ddots & \\
& & -J_{N-1} & I
\end{pmatrix}.$$

- *Forward* solves $\big(\partial g/\partial U\big)\,\tfrac{dU}{d\theta}=-\tfrac{\partial g}{\partial\theta}$ by forward-substitution, top down: the forward recurrence.
- *Reverse* solves the transpose $\big(\partial g/\partial U\big)^{\top}\lambda=\big(\tfrac{\partial L}{\partial U}\big)^{\top}$ by back-substitution, bottom up: the reverse recurrence.

The same triangular system, solved in opposite directions. (This is also the setup for §4, where an implicit solve makes $\partial g/\partial u$ a general matrix instead of triangular.)
:::

The payoff: **each operation only has to supply its own JVP and VJP.** AD chains them to differentiate the whole system, in whichever direction, with no gradient ever written for the full model.

Which to use, straight from the recurrences above:

- *Scalar loss, many parameters*: reverse wins (one pass, not $n$). This is training and inversion.
- *Few inputs, many outputs*: forward wins (one pass per input).

Reverse mode is the default for a scalar loss. On a small neural network, its recurrence is exactly backpropagation, matching `jax.grad`:

In [2]:
import jax.random as jr

# A small MLP: tanh hidden layers, linear output.  Params = list of (W, b).
sizes = [4, 16, 16, 1]
keys  = jr.split(jr.PRNGKey(0), len(sizes) - 1)
params = [(0.6 * jr.normal(k, (n_out, n_in)), jnp.zeros(n_out))
          for k, n_in, n_out in zip(keys, sizes[:-1], sizes[1:])]
x_in, y_tgt = jnp.array([0.3, -0.7, 1.1, 0.2]), jnp.array([0.5])

def mlp_loss(params, x):
    a = x
    for W, b in params[:-1]:
        a = jnp.tanh(W @ a + b)          # hidden layers
    W, b = params[-1]
    return jnp.sum((W @ a + b - y_tgt) ** 2)   # linear output + squared error

g_ad = jax.grad(mlp_loss)(params, x_in)        # reverse-mode AD

def hand_backprop(params, x):
    # forward pass, storing the states the backward pass will need
    a, acts, zs = x, [x], []
    for W, b in params[:-1]:
        z = W @ a + b; a = jnp.tanh(z); zs.append(z); acts.append(a)
    W, b = params[-1]; out = W @ a + b
    # backward pass = the adjoint recurrence  lambda_l = (dPhi_l/da_l)^T lambda_{l+1}
    lam = 2.0 * (out - y_tgt)                   # lambda = dL/d(output)
    grads = [(jnp.outer(lam, acts[-1]), lam)]   # output-layer gradients
    lam = params[-1][0].T @ lam                 # push cotangent into last hidden activation
    for l in range(len(params) - 2, -1, -1):
        delta   = (1.0 - jnp.tanh(zs[l]) ** 2) * lam      # through the tanh
        grads.insert(0, (jnp.outer(delta, acts[l]), delta))
        lam = params[l][0].T @ delta                      # (dPhi/da)^T = W^T (.)
    return grads

g_hand = hand_backprop(params, x_in)
worst = max(float(jnp.max(jnp.abs(a - b)))
            for (gw, gb), (hw, hb) in zip(g_ad, g_hand) for a, b in [(gw, hw), (gb, hb)])
print(f"max |AD - hand-coded backprop| over all weights & biases = {worst:.2e}")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


max |AD - hand-coded backprop| over all weights & biases = 4.44e-16


## 3 · A time-stepping solver is a composition

A solver that marches in time, $u_{k+1}=\Phi(u_k,\theta)$, is exactly a composition of steps, so nothing new is needed:

- The states are the time levels; the operations are the time steps.
- Reverse mode chains the steps' VJPs backward in time. This is the *discrete adjoint* of a time-dependent model, and it is what the glacier UDE uses.
- A shared parameter accumulates one gradient contribution per step (a weight reused everywhere gets a gradient summed over all its uses).

The only practical wrinkle is scale: a long solve has many steps, so storing every state for the backward pass is expensive. That is the checkpointing question of §5.

## 4 · When the state is an implicit solve: the adjoint method

The case that looks harder but is not. Sometimes the state is not produced by an explicit march; it is defined *implicitly*, as the solution of an equation:

$$g(u,\theta)=0 \quad\Longrightarrow\quad u=u(\theta).$$

- The examples are steady states, fixed points, and linear or nonlinear solves.
- $u(\theta)$ is still a function (implicit function theorem), and "solve for $u$" is still one link in the composition.
- The only new question: what is that link's VJP?

Differentiate the constraint, $\dfrac{\partial g}{\partial u}\dfrac{du}{d\theta}+\dfrac{\partial g}{\partial\theta}=0$, giving $\dfrac{du}{d\theta}=-\big(\tfrac{\partial g}{\partial u}\big)^{-1}\tfrac{\partial g}{\partial\theta}$. We never form that inverse. Given the cotangent $\bar u=\partial L/\partial u$ arriving at $u$, group the expensive factor into an adjoint variable $\lambda$:

$$\Big(\frac{\partial g}{\partial u}\Big)^{\!\top}\lambda = \Big(\frac{\partial L}{\partial u}\Big)^{\!\top}
\quad\text{(the adjoint equation)},\qquad
\frac{dL}{d\theta} = \frac{\partial L}{\partial\theta} - \lambda^{\top}\frac{\partial g}{\partial\theta}.$$

That is the classical adjoint method of PDE-constrained optimization (MacAyeal 1993). In the composition picture it is simply the VJP of the "solve" operation.

::: {.callout-important}
## The equivalence, in one place
Every link has a VJP, and reverse mode chains them. Only the work per link differs:

- an *explicit* link (a layer, a time step) has a matrix-vector VJP; chaining them is backprop (§2, §3);
- an *implicit* link (a steady-state solve) has a VJP that is itself a transposed linear solve, the adjoint equation.

For an explicit solver the system Jacobian is triangular, so the "solve" is free (substitution); for an implicit one it is a general matrix, so the adjoint is a genuine solve. Same chain rule, same reverse sweep; one link just does more.
:::

The efficiency is the point: forming $du/d\theta$ costs one solve per parameter, whereas the adjoint costs one transposed solve regardless of the number of parameters. On a small linear constraint $g(u,\theta)=Au-b(\theta)$:

In [5]:
# Constrained problem:  g(u, theta) = A u - b(theta) = 0  ->  u(theta) = A^{-1} b(theta).
# Loss L(u) = 1/2 ||u - u_target||^2.  DIRECT (one solve per parameter) vs ADJOINT (one solve).
n, p = 6, 5                                                     # state dim, number of parameters
Mn = jax.random.normal(jax.random.PRNGKey(0), (n, n))
A  = Mn @ Mn.T + n * jnp.eye(n)                                 # SPD system matrix
B  = jax.random.normal(jax.random.PRNGKey(1), (n, p))          # b(theta) = B @ theta
theta    = jax.random.normal(jax.random.PRNGKey(2), (p,))
u_target = jnp.ones(n)

def solve_u(theta): return jnp.linalg.solve(A, B @ theta)      # the forward solve, u(theta)
u    = solve_u(theta)
dLdu = u - u_target                                            # dL/du

# DIRECT / tangent-linear:  build du/dtheta column by column -> one linear solve PER PARAMETER
dudtheta    = jnp.stack([jnp.linalg.solve(A, B[:, j]) for j in range(p)], axis=1)   # p solves
grad_direct = dLdu @ dudtheta

# ADJOINT / cotangent:  ONE transposed solve, then dot products
lam          = jnp.linalg.solve(A.T, dLdu)                     # (dg/du)^T lambda = (dL/du)^T  <- 1 solve
grad_adjoint = lam @ B                                         # dL/dtheta = -lambda^T (dg/dtheta), dg/dtheta = -B

g_jax = jax.grad(lambda th: 0.5 * jnp.sum((solve_u(th) - u_target) ** 2))(theta)
print(f"parameters p = {p}")
print(f"  direct  : {p} linear solves   grad = {np.round(np.array(grad_direct), 5)}")
print(f"  adjoint : 1 linear solve      grad = {np.round(np.array(grad_adjoint), 5)}")
print(f"  adjoint matches direct: {bool(jnp.allclose(grad_direct, grad_adjoint))}"
      f"   |   matches jax.grad: {bool(jnp.allclose(grad_adjoint, g_jax))}")

parameters p = 5
  direct  : 5 linear solves   grad = [ 0.25687 -0.03022  0.0516  -0.32131  0.01129]
  adjoint : 1 linear solve      grad = [ 0.25687 -0.03022  0.0516  -0.32131  0.01129]
  adjoint matches direct: True   |   matches jax.grad: True


## 5 · Checkpointing: differentiating through long solves

::: {.callout-note}
## Common misconception
By default, neither JAX nor PyTorch checkpoints. The default reverse pass is maximum-memory, zero-recompute: save every intermediate, consume it in reverse.

- *PyTorch (eager):* saves activations; opt-in recompute is `torch.utils.checkpoint`.
- *JAX:* `jax.grad` saves residuals; opt-in rematerialization is `jax.checkpoint` (`jax.remat`).
- *Caveats:* XLA can rematerialize to fit a memory budget (heuristic), and `torch.compile`'s min-cut partitioner makes automatic save-vs-recompute choices; neither is the principled $O(\log N)$ schedule.
:::

Checkpointing trades memory for recomputation: store a few states, recompute the rest during the backward pass. The optimal schedule (Griewank and Walther's `revolve`) is $O(\log N)$ memory at $O(N\log N)$ compute.

::: {.callout-important}
## Why this matters for time-dependent inversions and UDEs
Training a UDE means backpropagating through every time step of the forward solve. Reverse mode through the steps (§3) needs each forward state $u_k$ to apply $(\partial\Phi_k/\partial u_k)^{\top}$, so storing all of them is $O(N)$ in the number of steps:

- our toy glacier: ~600 steps for a spin-up, so storing all is fine;
- a real inversion (Bolibar et al.: ~$10^5$ ODEs over a 2-D grid, many years): storing every state is infeasible.

Checkpointing keeps only $O(\log N)$ states and recomputes the rest forward, which is safe because the forward solve only ever runs forward in time. This is why the glacier solver passes `RecursiveCheckpointAdjoint`.
:::

Hand-rolled `revolve` on a chain of $N$ time steps, to see the memory and compute trade and confirm the gradient is unchanged:

In [6]:
N, theta = 64, 0.99
def phi(y):  return np.tanh(y) + theta * y
def dphi(y): return (1 - np.tanh(y) ** 2) + theta

def store_all(y0):
    ys = [y0]
    for _ in range(N): ys.append(phi(ys[-1]))
    lam = ys[N]
    for k in range(N - 1, -1, -1): lam = dphi(ys[k]) * lam
    return lam, len(ys), N

def checkpointed(y0, s):
    ckpts, y = {0: y0}, y0
    for k in range(1, N + 1):
        y = phi(y)
        if k % s == 0: ckpts[k] = y
    lam, fwd, peak = y, N, max(len(ckpts), s)
    for k in range(N - 1, -1, -1):
        if k in ckpts:
            yk = ckpts[k]
        else:
            base = max(c for c in ckpts if c < k); yk = ckpts[base]
            for _ in range(k - base): yk = phi(yk); fwd += 1
        lam = dphi(yk) * lam
    return lam, peak, fwd

g0, m0, f0 = store_all(1.0)
g1, m1, f1 = checkpointed(1.0, s=8)
print(f"store-all    : ~{m0:2d} states, {f0:3d} forward evals")
print(f"checkpointed : ~{m1:2d} states, {f1:3d} forward evals   (same gradient: {abs(g0-g1) < 1e-9})")

def deep(x):
    for _ in range(50): x = jnp.tanh(x) + 0.1 * x
    return jnp.sum(x ** 2)
x0 = jnp.linspace(-1, 1, 8)
print("jax.grad == jax.grad(jax.checkpoint(...)):", bool(jnp.allclose(jax.grad(deep)(x0), jax.grad(jax.checkpoint(deep))(x0))))

store-all    : ~65 states,  64 forward evals
checkpointed : ~ 9 states, 288 forward evals   (same gradient: True)


jax.grad == jax.grad(jax.checkpoint(...)): True


## 6 · Takeaways

- Write the model as a composition and chain the VJPs; that is reverse-mode AD, and for a scalar loss it is backprop.
- Each operation only needs its own JVP and VJP; AD assembles the gradient of the whole system.
- A time-stepping solver is such a composition, so backprop through time is the discrete adjoint.
- An implicit solve is the one link whose VJP is itself a linear solve, and that solve is the adjoint method of PDE-constrained optimization.

### References

- MacAyeal (1993), *Control methods in ice-sheet modeling*, J. Glaciol. 39(131).
- Griewank and Walther (2008), *Evaluating Derivatives* (SIAM); Griewank (1992), `revolve`, Optim. Methods Softw. 1.
- Baydin et al. (2018), *AD in machine learning: a survey*, JMLR 18(153).
- Chen et al. (2018), *Neural ODEs*, NeurIPS; Kidger (2021), *On Neural Differential Equations* (thesis; `diffrax`).
- Sapienza et al. (2024), *Differentiable programming for differential equations: a review*, arXiv:2406.09699.

---
*Composition-first companion to the UDE glaciology lecture.*